# Kaggriculture — v01 **“Drip”**

### Read the source, not the docs.

This notebook builds a Kaggriculture agent from an audit of the **environment source code**
(`kaggle_environments/envs/kaggriculture/kaggriculture.py`) rather than from the published
competition description — because the two disagree, and the disagreements are worth tens of
thousands of in-game dollars.

**What you get here**

1. A live, reproducible demonstration that the official docs are **wrong** on three points.
2. The market maths worked out numerically — how deep each product's demand actually is.
3. A strategy derived from those numbers, not from vibes.
4. A complete, commented `main.py` you can submit.
5. Local validation against the built-in agents.

**Headline finding:** the docs state *“Fertilizer can only be bought, not sold.”*
That is false. Fertilizer **is** sellable, is produced **free** by every animal every day,
and is the **only product the town never consumes** — so its ~493-unit demand pool is drained
by nothing except the two players. It is worth roughly **$25,000** per season.

---
## 0. Setup

`kaggle-environments` ships the Kaggriculture environment. We need it both to read the source
and to play local games.

In [ ]:
%%capture
!pip install -q --upgrade "kaggle-environments>=1.32.2"

In [ ]:
import inspect, math, re, textwrap
from kaggle_environments import make
from kaggle_environments.envs.kaggriculture import kaggriculture as KG

env = make("kaggriculture", debug=True)
print("environment :", env.name, "v" + str(env.version))
print("agents      :", env.specification.agents)
print("episodeSteps:", env.configuration.episodeSteps,
      "| turnsPerDay:", env.configuration.turnsPerDay)
print("source      :", inspect.getsourcefile(KG))

---
## 1. What actually wins this competition

This is a **simulation ladder**, not a modelling competition. There is no train/test split and no
`submission.csv`. A few properties of the scoring drive every design decision:

| Fact | Consequence |
|---|---|
| Ranking is **win / loss / tie only** — the coin margin is discarded | Optimise **P(win)**, not expected coins. Kill variance. |
| Matchmaking pairs you with **similar-rated** agents | Beating `starter` proves nothing. The bar is other tuned bots. |
| Only your **latest 2 submissions** stay active | Don't spam. Every submission is a real decision. |
| A **validation episode** runs your agent against itself | **Any uncaught exception kills the submission.** Defensive code is a scoring feature. |
| **Ten** prizes of \$5,000, all equal | Reliable top-10 beats a high-variance moonshot. |

There is also a hard compute budget most write-ups miss:

In [ ]:
import json
spec = json.loads(open(inspect.getsourcefile(KG).replace(".py", ".json")).read())
cfg = spec["configuration"]
for k in ("episodeSteps", "actTimeout", "runTimeout", "agentTimeout"):
    if k in cfg:
        print(f"{k:14}: {cfg[k]}")
print("remainingOverageTime:", spec["observation"]["remainingOverageTime"])

**`actTimeout = 1` second per turn, with only 60 seconds of overage for the entire 720-turn
episode.** That rules out per-turn search, MCTS, or any neural inference. The winning shape is a
fast heuristic scheduler. Our agent averages well under 10 ms per turn.

---
## 2. Three places the documentation is wrong

Let's not take my word for it — read the code.

### 2.1 Fertilizer **is** sellable

`AGENTS.md` says: *"Fertilizer can only be bought, not sold."*

In [ ]:
src = inspect.getsource(KG)

# The SELL whitelist is the PRODUCTS list. Does it contain FERTILIZER?
print("PRODUCTS =", KG.PRODUCTS)
print("FERTILIZER in the SELL whitelist ->", "FERTILIZER" in KG.PRODUCTS)
print()
for line in src.splitlines():
    if ('op == "SELL"' in line) or ('op == "BUY_PRODUCT"' in line):
        print(line.strip())

`SELL` accepts **any** member of `PRODUCTS`, and `PRODUCTS` includes `FERTILIZER`.
Only `BUY_PRODUCT` is restricted to `("WHEAT", "FERTILIZER")`.

Now the part that makes it valuable — **who consumes fertilizer?**

In [ ]:
print("Town centre consumes:", KG.TOWN_CENTER_PRODUCTS)
print("FERTILIZER excluded from town centre ->", "FERTILIZER" not in KG.TOWN_CENTER_PRODUCTS)
print()
print("Shops and what they demand:")
for shop, items in KG.SHOPS.items():
    print(f"  {shop:16} {items}")
print()
demanded = {i for items in KG.SHOPS.values() for i in items}
print("Any shop demanding FERTILIZER ->", "FERTILIZER" in demanded)

So fertilizer has **zero** demand-side drain. Its market inventory moves only when a *player*
buys or sells. Combined with `COLLECT_FERTILIZER` producing one unit per animal per day **for
free**, this is the single most under-priced lane in the game — and it is a **race**, because the
pool is shared with the opponent and the price only falls.

### 2.2 `CARE` banks **+1** per day, not +2

The docs say *"`pending_care_bonus` increments by 2"*.

In [ ]:
for line in src.splitlines():
    if "pending_care_bonus" in line:
        print(line.rstrip())

The accrual line is `+ 1`. Every animal yield estimate in the docs is therefore **double** the
truth. Real steady-state output with daily FEED + CARE:

| Animal | Interval | Per production | Per day |
|---|---:|---:|---:|
| Goose | 1 day | 1 + 1 = 2 | **2.00 egg** |
| Cow | 2 days | 1 + 2 = 3 | **1.50 milk** |
| Sheep | 3 days | 1 + 3 = 4 | **1.33 wool** |

Note also two traps in that same function: the care bank is **wiped** if the animal is unfed on a
production day, and `CARE` only accrues on days the animal was **also fed**.

### 2.3 Fertilizer is available every day, with no `CARE` requirement

The docs' tile reference says `fertilizer_available` is *"set after CARE"*.

In [ ]:
fn = inspect.getsource(KG._daily_refresh_animals)
print(fn[fn.index("a = ANIMALS"):])

`tile["fertilizer_available"] = True` is unconditional — it runs for every animal that survived
the day, regardless of `cared_today` or `fed_today`. Free money, one action to collect.

### 2.4 One more that will kill your crops

`_new_plant` starts a plant at `consecutive_unwatered = 1`, and two consecutive unwatered days
turns it into a weed. **A crop not watered on its own planting day dies that night.**

In [ ]:
p = inspect.getsource(KG._new_plant)
print(p)

---
## 3. The market is the whole game

Price moves with market inventory:

$$\text{price}(inv) = base \;\pm\; \underbrace{\frac{target \cdot base}{f(T)}}_{\text{amp}} \cdot f\big(|inv - I_0|\big)$$

with an independent shape function $f \in \{linear, sq, sqrt, log\}$ on each side. Selling pushes
inventory **up** and price **down**. The choice of $f$ per product is what makes them play
completely differently.

In [ ]:
print(f"{'item':12}{'base':>6}{'above_func':>12}{'above_target':>14}")
for k, v in KG.MARKET_PARAMS.items():
    print(f"{k:12}{v['base']:>6}{v['above_func']:>12}{v['above_target']:>14}")

### 3.1 How much can you actually sell?

Walk the curve forward one unit at a time until the price hits the \$1 floor. That number is the
product's **glut depth** — and it is startlingly small for the premium goods.

In [ ]:
def depth_table():
    rows = []
    for item in KG.MARKET_PARAMS:
        inv, rev, n = KG.MARKET_I0, 0, 0
        while n < 30000:
            p = KG.market_price(item, inv)
            if p <= 1:
                break
            rev += p; inv += 1; n += 1
        rows.append((item, KG.MARKET_PARAMS[item]["base"], n, rev, rev // max(n, 1)))
    return rows

print(f"{'product':12}{'base':>6}{'units to $1':>13}{'total $':>12}{'avg $':>7}")
print("-" * 50)
for item, base, n, rev, avg in sorted(depth_table(), key=lambda r: -r[3]):
    tag = "  <- effectively unlimited (log curve)" if n >= 20000 else ""
    print(f"{item:12}{base:>6}{n:>13,}{rev:>12,}{avg:>7}{tag}")

Read that carefully:

* **Wool dies after 59 units. Milk after 76. Strawberry after 62.** Dump a single big harvest and
  you have destroyed the price for the rest of the season.
* **Melon** is the biggest single reservoir — \$26k at an average of \$167/unit — but only 158
  units deep.
* **Fertilizer** is \$25k at \$50/unit average, and (per §2.1) nothing but the players touches it.
* **Egg and wheat use `log` curves and never crash.** They are infinite sinks, but cheap ones.

### 3.2 Town demand refills the well every day

Shops consume every 4 steps (6×/day); the town centre every 12 steps (2×/day), scaling ×2 from
day 10 and ×4 from day 20. That drain is what makes premium goods sellable *repeatedly*.

In [ ]:
from collections import Counter
drain = Counter()
for shop, items in KG.SHOPS.items():
    mult = 2 if len(items) == 1 else 1          # single-product shops buy double
    for it in items:
        drain[it] += mult * 6                    # 6 shop ticks per day
for it in KG.TOWN_CENTER_PRODUCTS:
    drain[it] += 2 * 4                           # 2 centre ticks/day x4 after day 20

print(f"{'product':12}{'drain/day':>11}{'$/day at base':>16}")
print("-" * 39)
for it, n in drain.most_common():
    print(f"{it:12}{n:>11}{n * KG.MARKET_PARAMS[it]['base']:>16,}")
print(f"{'FERTILIZER':12}{0:>11}{'--  never consumed':>16}")

**This is the central strategic insight of the game.**

Premium goods have *tiny* instantaneous depth but *large* daily replenishment. The correct policy
is therefore not "produce and sell" but **produce and drip** — sell down to a price floor each
day and let town demand refill the well overnight. An agent that dumps its harvest converts
\$4,000/day of milk headroom into \$76 of one-off revenue and a dead market.

Our whole sell policy is one function: *how many units can I sell before the price drops below my
floor?*

---
## 4. Strategy

Combining yields (§2.2), depth (§3.1) and drain (§3.2), and charging every action its
opportunity cost, the lanes rank like this:

| Lane | $/productive action | Limited by |
|---|---:|---|
| **Melon** | **\$62–90** | 158-unit depth, 8/day drain |
| **Sheep → wool** | **\$55** | 20/day drain → ~11 sheep |
| **Cow → milk** | **\$50** | 26/day drain → ~13 cows |
| Fertilized own wheat | \$36–43 | rises as market wheat inflates |
| Strawberry | \$14–28 | 62-unit depth; only 4 productions/tile |
| Goose → egg | \$5–22 | uncapped, but feed cost eats the margin |
| Carrot | \$13 | weak |

A note on **geese**, because it is counter-intuitive: eggs never crash, which makes them look like
a free lunch. They aren't. Every animal eats **1 wheat/day**, and wheat inflates as
`25 + √(units drained)` — past \$65 once the town and both players have pulled ~1,600 units. A
goose then earns 2 eggs (~\$74) against a ~\$50 feed bill for 5 actions. **That is why the top
public agent runs zero geese**, and we follow it.

**The plan:**

1. Buy **sheep first** (earliest premium yield, day 6), then cows.
2. Plant **melon far from the shed** — it only needs water — and wheat near it.
3. `COLLECT_FERTILIZER` from every animal, every day, and race into the fertilizer pool early.
4. **Buy** feed wheat rather than growing it (market orders cost **zero unit-actions**), until
   wheat inflates past our threshold.
5. **Meter every sale** against a price floor. Never dump.
6. Water on the planting day, then only every other day outside the bonus window — this alone
   saves hundreds of actions per season.

### Labour is nearly free — and that is the real bottleneck

Hiring costs `fib(n)` for the n-th hire of the day, resetting nightly:

In [ ]:
cum = 0
print(f"{'hands':>6}{'this hire':>11}{'cumulative/day':>16}")
for n in range(0, 18):
    c = KG._hire_cost(n)
    cum += c
    if n + 1 in (3, 6, 9, 12, 14, 15, 16, 17, 18):
        print(f"{n+1:>6}{c:>11}{cum:>16,}")

**12 hands cost \$376 for the whole day** — 312 unit-actions for pocket change. Labour is not the
constraint; *useful* labour is. The captured leaderboard agent spends **55% of all its hand
actions just walking**. Our scheduler charges every job its travel distance and places animals
near the shed (they need daily wheat) and melons far from it (they only need water).

---
## 5. The agent

Design: no search, no learning — a **value-per-action scheduler**.

Each turn we
1. scan the farm into a flat typed view,
2. emit a **job** for every actionable tile, tagged with the dollar value of doing it
   (`HARVEST 4 wool` = \$800; `CARE sheep` = \$200 because it banks +1 wool; `WATER melon that
   dies tonight` = the whole plant),
3. assign units greedily by `value − travel × opportunity_cost`,
4. plan the market queue, ordered high-value-first because order slots are consumed against the
   opponent's queue.

Everything is wrapped in a total `try/except` — a crash forfeits the episode.

In [ ]:
%%writefile main.py
"""
Kaggriculture agent — v01 "Drip"

Strategy in one line:
    Saturate the highest $/action lanes (melon, sheep, cow), harvest the free
    fertilizer byproduct from every animal, and METER every sale against the
    live price curve instead of dumping.

All game constants below are transcribed from the environment source
(kaggle_environments/envs/kaggriculture/kaggriculture.py) and NOT from the
published docs, which are wrong on several decisive points:

  * FERTILIZER *is* sellable  (PRODUCTS L25 includes it; SELL whitelist L571).
    The town never consumes it, so its 493-unit pool is drained only by the
    two players -> a first-mover race worth ~$25k.
  * CARE banks +1/day, not +2   (L800).
  * fertilizer_available is set every end-of-day for every surviving animal,
    with no CARE dependency     (L801).
  * A plant not watered on its PLANTING DAY dies that night
    (_new_plant seeds consecutive_unwatered = 1).
  * Animals have no lifespan and no production cap; max_held is storage only.

Hard runtime limit: actTimeout = 1s/turn, 60s total overage for the episode.
This file is pure stdlib and does no search; typical turn cost is < 5 ms.
"""

import math
import os

# ----------------------------------------------------------------------------
# Verified game constants
# ----------------------------------------------------------------------------

CROPS = {
    "WHEAT":      {"seed": 10,  "first_yield_day": 2,  "max_yield_day": 4,  "interval": 0, "max_yield": 6, "ongoing": False, "product": "WHEAT"},
    "CARROT":     {"seed": 20,  "first_yield_day": 2,  "max_yield_day": 3,  "interval": 0, "max_yield": 4, "ongoing": False, "product": "CARROT"},
    "TOMATO":     {"seed": 50,  "first_yield_day": 8,  "max_yield_day": 8,  "interval": 1, "max_yield": 4, "ongoing": True,  "product": "TOMATO"},
    "STRAWBERRY": {"seed": 100, "first_yield_day": 10, "max_yield_day": 10, "interval": 2, "max_yield": 4, "ongoing": True,  "product": "STRAWBERRY"},
    "MELON":      {"seed": 80,  "first_yield_day": 10, "max_yield_day": 12, "interval": 0, "max_yield": 6, "ongoing": False, "product": "MELON"},
}

ANIMALS = {
    "GOOSE": {"cost": 300, "structure": "COOP",    "first_yield_day": 4, "interval": 1, "max_held": 4, "product": "EGG"},
    "COW":   {"cost": 400, "structure": "PASTURE", "first_yield_day": 8, "interval": 2, "max_held": 6, "product": "MILK"},
    "SHEEP": {"cost": 500, "structure": "PASTURE", "first_yield_day": 6, "interval": 3, "max_held": 6, "product": "WOOL"},
}

PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON",
            "EGG", "MILK", "WOOL", "FERTILIZER"]

MARKET_I0 = 10000
PRICE_FLOOR = 1

MARKET_PARAMS = {
    "WHEAT":      {"base":  25, "I0": MARKET_I0, "T": 400, "below_func": "sqrt",   "below_target": 0.80, "above_func": "log",    "above_target": 0.20},
    "CARROT":     {"base":  35, "I0": MARKET_I0, "T": 450, "below_func": "log",    "below_target": 0.20, "above_func": "sqrt",   "above_target": 0.70},
    "TOMATO":     {"base":  60, "I0": MARKET_I0, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "sqrt",   "above_target": 0.60},
    "STRAWBERRY": {"base": 120, "I0": MARKET_I0, "T": 100, "below_func": "sqrt",   "below_target": 0.70, "above_func": "linear", "above_target": 1.60},
    "MELON":      {"base": 250, "I0": MARKET_I0, "T": 300, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.60},
    "EGG":        {"base":  50, "I0": MARKET_I0, "T": 332, "below_func": "linear", "below_target": 0.40, "above_func": "log",    "above_target": 0.20},
    "MILK":       {"base": 160, "I0": MARKET_I0, "T": 122, "below_func": "sqrt",   "below_target": 0.60, "above_func": "linear", "above_target": 1.60},
    "WOOL":       {"base": 200, "I0": MARKET_I0, "T": 105, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.20},
    "FERTILIZER": {"base": 100, "I0": MARKET_I0, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "linear", "above_target": 0.40},
}

SHOPS = {
    "BAKERY":         ["EGG", "WHEAT"],
    "PIZZA_SHOP":     ["MILK", "TOMATO", "WHEAT"],
    "BRUNCH_SPOT":    ["EGG", "WHEAT", "STRAWBERRY"],
    "YARN_STORE":     ["WOOL"],
    "ICE_CREAM_SHOP": ["STRAWBERRY", "MILK", "WHEAT"],
    "PET_CAFE":       ["CARROT"],
    "SMOOTHIE_SHOP":  ["STRAWBERRY", "MILK"],
    "FARMERS_MARKET": ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY"],
}

LAND_ORDER = ["NE", "SW", "SE"]
LAND_PRICES = [1000, 2000, 4000]

TURNS_PER_DAY = 24
LAST_DAY = 29
SHED_CAPACITY = 100
MAX_MARKET_ORDERS = 10

MOVE_OPS = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}


# ----------------------------------------------------------------------------
# Tunable policy config  (this is the parameter search space)
# ----------------------------------------------------------------------------

class CFG:
    # --- labour ---------------------------------------------------------
    HAND_BASE = 4               # hands even with an empty farm
    HAND_PER_ANIMAL = 0.34      # ~3 animals serviced per unit per day
    HAND_PER_PLANT = 0.10       # extra hands per living plant
    HAND_MAX = int(os.environ.get("KAGG_HANDS", "12"))               # fib(15)=$987 for the 16th hire
    HAND_MONEY_FRAC = 0.10      # never spend >10% of bank on a day's hires

    # --- capital allocation ---------------------------------------------
    # Animal count is NOT a fixed cap. It is bounded by (a) how many units we
    # can afford to work them and (b) whether the product still clears above
    # our price floor. Measured episodes end with milk/wool at ~2x base, i.e.
    # the market is starved: production, not demand depth, is the constraint.
    ANIMALS_PER_UNIT = float(os.environ.get("KAGG_APU", "2.2"))      # animals one unit can service per day
    ANIMAL_BUY_PRICE_FRAC = 0.80   # stop buying a species below this x base
    # Geese are deliberately excluded: ~$5-22 per action once wheat inflates,
    # and each one needs BUILD_COOP + PICKUP + PLACE before it earns anything.
    # In testing the agent bought 36 and managed to place 2.
    SPECIES_ORDER = ("SHEEP", "COW")
    # never hold more unplaced animals than we can house shortly
    ANIMAL_BACKLOG = 3
    # Herd target = (daily town demand) / (yield per animal per day) x ratio.
    # Slightly under-supplying keeps the price at or above base; over-supplying
    # crashes it (28 cows drove milk from $160 to $24 in testing).
    SUPPLY_RATIO = {"MILK": 0.95, "WOOL": 0.95, "EGG": 2.20}
    # An animal that starves is a total loss, so every purchase must also
    # reserve its feed. This is what stops the herd outrunning the bank.
    FEED_BUFFER_PER_ANIMAL = 280
    MAX_BUY_PER_DAY = int(os.environ.get("KAGG_MBD", "4"))         # ramp the herd instead of spiking it
    # Crop tile budget. STRAWBERRY is the single best lane in the game and is
    # easy to miss: the town drains 32/day of it (more than any other product),
    # so a farm of this size never gluts it and it clears at ~2x base all
    # season. It also needs no feed and no daily FEED/CARE, unlike animals.
    STRAWBERRY_TILES = int(os.environ.get("KAGG_STRAW", "16"))
    MELON_TILES = int(os.environ.get("KAGG_MEL", "12"))            # 2 cycles; metered against its 8/day drain
    WHEAT_TILES = 6             # fast cash (first yield day 2) + feed
    CASH_RESERVE = 300          # floor under the bank at all times
    # Never sink the whole bank into slow seed. Strawberry costs $100 and pays
    # nothing until day 10; buying 30 on day 0 spends the entire starting bank
    # and leaves nothing for land, which is the real multiplier.
    SEED_BUDGET_FRAC = float(os.environ.get("KAGG_SEEDF", "0.80"))
    SEED_TILE_LOOKAHEAD = 4     # buy at most this many seeds beyond free tiles

    LAND_BUY_MONEY = [1200, 2400, 5000]   # bank needed before each BUY_LAND
    LAND_BUY_DAY = [0, 3, 8]              # earliest day for each BUY_LAND

    # last day it is still worth starting each investment
    LAST_DAY_BUY = {"SHEEP": 23, "COW": 21, "GOOSE": 24}
    # strawberry fires productions at +10/+12/+14/+16 days
    LAST_DAY_PLANT_STRAWBERRY = 19
    LAST_DAY_PLANT_MELON = 19
    LAST_DAY_PLANT_WHEAT = 26

    # --- wheat / feed ----------------------------------------------------
    # CRITICAL: the shed holds only 100 items TOTAL, shared between feed wheat
    # and unsold produce, and end-of-day overflow is DESTROYED. Hoarding feed
    # silently bins every harvest. So we buy wheat just-in-time -- market
    # orders cost zero unit-actions, so re-buying every turn is free.
    WHEAT_BUY_MAX_PRICE = 110   # feed is cheap next to a $300 milk cheque
    WHEAT_SPARE = 8             # shed wheat target = unfed animals + this
    WHEAT_PICKUP = 12           # bigger loads = fewer shed round-trips
    SHED_SOFT_LIMIT = 52        # above this, start discounting sell floors
    SHED_PANIC = 88             # above this, dump at any price

    # --- market metering: sell only while price >= frac * base ----------
    # Floors are expressed as a fraction of base price. Setting them AT base
    # (1.0) is the key to the whole economy: the town drains the market every
    # turn, so price recovers above base continuously and we sell only into
    # that recovery -- roughly one unit per turn per product, which is exactly
    # the drain rate. Selling below base means pushing inventory past I0, and
    # the premium curves are savage there (wool and melon are worth $1 after
    # ~59 and ~158 units of glut). Measured: floors at 0.5-0.62 ended the
    # season with melon at $31 and milk at $24; at 1.0 they hold ~$250.
    SELL_FLOOR = {
        "MELON": 0.66, "WOOL": 0.66, "MILK": 0.62, "STRAWBERRY": 0.60,
        "EGG": 0.75, "FERTILIZER": 0.40, "WHEAT": 0.76,
        "CARROT": 0.70, "TOMATO": 0.70,
    }
    # sweep hook: scale every floor at once for offline tuning
    FLOOR_SCALE = float(os.environ.get("KAGG_FLOOR", "1.0"))
    LIQUIDATE_DAY = 28          # from here, floors decay toward 1
    FERT_USE_BELOW = 45         # if fertilizer sells under this, use it instead
    # Sell no faster than the town consumes. >1 leans on the instantaneous
    # depth as well as the daily drain.
    SELL_RATE_MULT = float(os.environ.get("KAGG_RATE", "6.0"))
    FERT_SELL_PER_DAY = 18      # fertilizer has no town demand at all

    # --- scheduling ------------------------------------------------------
    TRAVEL_COST = 20            # $ of opportunity cost per turn spent walking
    SHED_DEPOSIT_LOAD = 14      # carry this many items before a shed run
    DIG_VALUE = 210.0           # a blocked tile is a lost crop
    WATER_INSURANCE = 0.35      # x product price, for a not-yet-critical water
    # Don't plant more than the labour force can keep alive. Each plant costs
    # roughly this many actions per day (water on alternate days, plus its
    # share of harvest/fertilize/travel); each animal ~3.5.
    ACTIONS_PER_PLANT = 1.15
    ACTIONS_PER_ANIMAL = 3.6
    PRODUCTIVE_FRAC = float(os.environ.get("KAGG_PF", "0.62"))      # share of unit-turns that are not walking
    CAPACITY_SLACK = 0.88       # only plant up to this share of capacity


# ----------------------------------------------------------------------------
# Market maths (exact replica of the environment's price function)
# ----------------------------------------------------------------------------

def _shape(func, x):
    x = max(0.0, x)
    if func == "linear":
        return x
    if func == "sq":
        return x * x
    if func == "sqrt":
        return math.sqrt(x)
    if func == "log":
        return math.log(1.0 + x)
    if func == "log10":
        return math.log10(1.0 + x)
    return x


def market_price(item, inventory):
    """Mirror of kaggriculture.market_price (L177-191)."""
    p = MARKET_PARAMS[item]
    base, i0, t = p["base"], p["I0"], p["T"]
    if inventory < i0:
        f = p["below_func"]
        amp = p["below_target"] * base / _shape(f, t)
        price = base + amp * _shape(f, i0 - inventory)
    else:
        f = p["above_func"]
        amp = p["above_target"] * base / _shape(f, t)
        price = base - amp * _shape(f, inventory - i0)
    return max(PRICE_FLOOR, int(round(price)))


def drain_per_day(product, unlocked_shops, day, full=False):
    """Units of `product` the town removes from the market each day.

    Shops tick every 4 steps (6x/day) and take 1 of each product they demand,
    doubled for single-product shops. The town centre ticks every 12 steps
    (2x/day) and scales x2 from day 10 and x4 from day 20.

    `full=True` prices in every shop as if already unlocked - used for herd
    planning, because an animal bought today only starts producing in 6-8 days
    by which time more shops will have opened.
    """
    shops = SHOPS.keys() if full else (unlocked_shops or [])
    n = 0
    for shop in shops:
        items = SHOPS.get(shop)
        if items and product in items:
            n += (2 if len(items) == 1 else 1) * 6
    if product != "FERTILIZER":
        mult = 4 if (full or day >= 20) else (2 if day >= 10 else 1)
        n += mult * 2
    return n


def units_above_floor(item, inventory, floor, cap=400):
    """How many units can be sold before the price drops below `floor`.

    Selling adds one unit to market inventory at a time, so this walks the
    curve forward. This is the whole of our sell discipline: premium goods
    have only 59-76 units of depth but are replenished by town demand every
    day, so we drip to the floor and let the town refill overnight.
    """
    n = 0
    inv = inventory
    while n < cap:
        if market_price(item, inv) < floor:
            break
        n += 1
        inv += 1
    return n


# ----------------------------------------------------------------------------
# Small helpers
# ----------------------------------------------------------------------------

def _tile_at(tiles, x, y):
    try:
        return tiles[y][x]
    except Exception:
        return "LOCKED"


def _is_dict(v):
    return isinstance(v, dict)


def _dist(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def _step_toward(fx, fy, tx, ty):
    """One orthogonal step. LOCKED tiles are passable so no pathfinding."""
    if fx < tx:
        return "EAST"
    if fx > tx:
        return "WEST"
    if fy < ty:
        return "SOUTH"
    if fy > ty:
        return "NORTH"
    return None


def _shed_ports(board_size):
    half = board_size // 2
    return [(half - 1, half - 1), (half, half - 1),
            (half - 1, half), (half, half)]


def _fib(n):
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def _hire_cost(n_already_today):
    return _fib(n_already_today)


# ----------------------------------------------------------------------------
# Per-episode state
# ----------------------------------------------------------------------------

_STATE = {}


def _state(player):
    if player not in _STATE:
        _STATE[player] = {"last_step": -1}
    return _STATE[player]


# ----------------------------------------------------------------------------
# Farm scan
# ----------------------------------------------------------------------------

class Farm(object):
    """Flattened, typed view of everything we need from the observation."""

    def __init__(self, obs):
        self.player = obs.get("player", 0)
        me = obs["farms"][self.player]
        self.money = float(me.get("money", 0))
        self.tiles = me["tiles"]
        self.board = len(self.tiles)
        self.farmer = list(me.get("farmer", [0, 0]))
        self.hands = [list(h) for h in (me.get("hands") or [])]
        self.hires_today = int(me.get("hires_today", 0))
        self.quadrants = list(me.get("unlocked_quadrants") or ["NW"])

        priv = obs.get("private", {}) or {}
        self.shed = dict(priv.get("shed", {}) or {})
        self.seeds = dict(priv.get("seeds", {}) or {})
        self.inventories = [dict(i or {}) for i in (priv.get("inventories") or [{}])]

        mk = obs.get("market", {}) or {}
        self.inv = dict(mk.get("inventory", {}) or {})
        self.prices = dict(mk.get("prices", {}) or {})
        town = obs.get("town", {}) or {}
        self.shops = list(town.get("unlocked_shops") or [])

        self.day = int(obs.get("day", 0))
        self.hour = int(obs.get("hour", 0))
        self.step = int(obs.get("step", self.day * TURNS_PER_DAY + self.hour))

        self.ports = [p for p in _shed_ports(self.board)
                      if _tile_at(self.tiles, p[0], p[1]) != "LOCKED"]
        if not self.ports:                       # defensive; NW always unlocked
            self.ports = [(self.board // 2 - 1, self.board // 2 - 1)]

        # --- classify every tile once -----------------------------------
        self.empty = []            # unlocked, nothing on it
        self.plants = []           # (x, y, tile)
        self.weeds = []
        self.animals = []          # (x, y, tile) occupied structures
        self.free_struct = []      # (x, y, tile) built but empty
        for y in range(self.board):
            row = self.tiles[y]
            for x in range(self.board):
                t = row[x]
                if t == "LOCKED":
                    continue
                if t is None:
                    self.empty.append((x, y))
                elif _is_dict(t):
                    kind = t.get("kind")
                    if kind == "PLANT":
                        self.plants.append((x, y, t))
                    elif kind == "WEED":
                        self.weeds.append((x, y))
                    elif kind in ("COOP", "PASTURE"):
                        if t.get("animal"):
                            self.animals.append((x, y, t))
                        else:
                            self.free_struct.append((x, y, t))

        self.n_animals = len(self.animals)
        self.counts = {"SHEEP": 0, "COW": 0, "GOOSE": 0}
        for _, _, t in self.animals:
            a = t.get("animal")
            if a in self.counts:
                self.counts[a] += 1
        # animals bought but not yet placed still count against the caps
        for a in self.counts:
            self.counts[a] += int(self.shed.get(a, 0))
            for inv in self.inventories:
                self.counts[a] += int(inv.get(a, 0))

        self.crop_tiles = {}
        for _, _, t in self.plants:
            c = t.get("crop")
            self.crop_tiles[c] = self.crop_tiles.get(c, 0) + 1

        self.shed_used = sum(int(v) for v in self.shed.values())

    # -- convenience -----------------------------------------------------
    def price(self, item):
        if item in self.prices:
            return max(1, int(self.prices[item]))
        return market_price(item, self.inv.get(item, MARKET_I0))

    def can_absorb(self, product, n):
        """Is there room in BOTH the shed and the market for n more units?"""
        carried = sum(int(i.get(product, 0)) for i in self.inventories)
        if self.shed_used + carried + n > CFG.SHED_SOFT_LIMIT + 20:
            return False
        # Deliberately a SHED check only. Gating harvest on the sell price too
        # was fatal: with a floor at base, ripe melons were never harvested at
        # all and rotted on the plant (income did not start until day 21).
        # Standing crop is safe until decay; the shed is the real hazard.
        return True

    def port_dist(self, x, y):
        return min(_dist((x, y), p) for p in self.ports)

    def nearest_port(self, x, y):
        return min(self.ports, key=lambda p: _dist((x, y), p))

    def unit_pos(self, i):
        return self.farmer if i == 0 else self.hands[i - 1]

    def unit_inv(self, i):
        return self.inventories[i] if i < len(self.inventories) else {}

    @property
    def n_units(self):
        return 1 + len(self.hands)


# ----------------------------------------------------------------------------
# Sell floors
# ----------------------------------------------------------------------------

def _sell_floor(farm, item):
    base = MARKET_PARAMS[item]["base"]
    frac = CFG.SELL_FLOOR.get(item, 0.6) * CFG.FLOOR_SCALE
    floor = base * frac

    # Shed pressure: every item still sitting in the shed at nightfall risks
    # pushing the day's harvest over the 100-item cap, where it is destroyed.
    # A discounted sale always beats an incinerated one.
    used = farm.shed_used
    if used > CFG.SHED_SOFT_LIMIT:
        if used >= CFG.SHED_PANIC:
            return 1.0
        span = float(CFG.SHED_PANIC - CFG.SHED_SOFT_LIMIT)
        floor *= max(0.05, 1.0 - (used - CFG.SHED_SOFT_LIMIT) / span)

    # End-of-season liquidation: anything unsold at step 718 is worth zero.
    if farm.day >= CFG.LIQUIDATE_DAY:
        decay = (farm.day - CFG.LIQUIDATE_DAY) * TURNS_PER_DAY + farm.hour
        span = float((LAST_DAY - CFG.LIQUIDATE_DAY + 1) * TURNS_PER_DAY)
        floor = floor * max(0.0, 1.0 - decay / span)
    return max(1.0, floor)


# ----------------------------------------------------------------------------
# Market planning
# ----------------------------------------------------------------------------

def plan_market(farm, state):
    """Build this turn's ordered market queue (max 10 slots).

    Slot order matters: orders are consumed slot-by-slot against the
    opponent's queue, so the most valuable sales go first.
    """
    orders = []
    money = farm.money
    day = farm.day

    # --- 1. land ---------------------------------------------------------
    n_extra = len(farm.quadrants) - 1
    if n_extra < len(LAND_PRICES):
        need = LAND_PRICES[n_extra]
        if (money >= max(need, CFG.LAND_BUY_MONEY[n_extra])
                and day >= CFG.LAND_BUY_DAY[n_extra]
                and day <= 24):
            orders.append(["BUY_LAND"])
            money -= need

    # --- 2. labour -------------------------------------------------------
    want = (CFG.HAND_BASE
            + CFG.HAND_PER_ANIMAL * farm.n_animals
            + CFG.HAND_PER_PLANT * len(farm.plants))
    want = int(min(CFG.HAND_MAX, max(1, round(want))))
    if farm.hour <= 2 and farm.hires_today < want:
        budget = money * CFG.HAND_MONEY_FRAC
        spent = 0.0
        n = farm.hires_today
        while n < want and len(orders) < 6:
            c = _hire_cost(n)
            if spent + c > budget or c > money - spent:
                break
            orders.append(["HIRE"])
            spent += c
            n += 1
        money -= spent

    # --- 3. metered sells (highest value first) --------------------------
    sells = []
    unfed_now = sum(1 for (_, _, t) in farm.animals if not t.get("fed_today"))
    wheat_reserve = unfed_now + CFG.WHEAT_SPARE
    for item in PRODUCTS:
        have = int(farm.shed.get(item, 0))
        if have <= 0:
            continue
        if item == "WHEAT":
            have -= wheat_reserve
            if have <= 0:
                continue
        if item == "FERTILIZER":
            # keep a couple for melon fertilising when fertilizer is cheap
            if farm.price("FERTILIZER") < CFG.FERT_USE_BELOW:
                have -= 4
                if have <= 0:
                    continue
        inv = int(farm.inv.get(item, MARKET_I0))
        floor = _sell_floor(farm, item)
        n = units_above_floor(item, inv, floor, cap=min(have, 300))

        # RATE LIMIT -- the heart of the strategy. A price floor alone does not
        # stop a dump: sixteen ripe melons clear the floor check and go out in
        # ONE order, tanking the price. The town only consumes so much per
        # turn, so we sell at roughly that rate and let demand refill the well
        # between turns. Without this the floor value makes no difference at
        # all (measured: identical results from 0.45x to 0.75x base).
        if item != "WHEAT" and farm.day < CFG.LIQUIDATE_DAY:
            per_day = drain_per_day(item, farm.shops, farm.day)
            if item == "FERTILIZER":
                # nothing in town eats fertilizer, so pace it against the pool
                per_day = CFG.FERT_SELL_PER_DAY
            rate = max(1, int(math.ceil(per_day / float(TURNS_PER_DAY)
                                        * CFG.SELL_RATE_MULT)))
            n = min(n, rate)
        if n > 0:
            sells.append((farm.price(item) * n, item, n))
    sells.sort(reverse=True)
    for _, item, n in sells:
        orders.append(["SELL", item, int(n)])

    # --- 4. feed wheat (just-in-time; see WHEAT_SPARE note in CFG) --------
    wheat_target = unfed_now + CFG.WHEAT_SPARE
    wheat_have = int(farm.shed.get("WHEAT", 0))
    for inv in farm.inventories:
        wheat_have += int(inv.get("WHEAT", 0))
    if farm.n_animals > 0 and wheat_have < wheat_target:
        wprice = market_price("WHEAT", int(farm.inv.get("WHEAT", MARKET_I0)) - 1)
        if wprice <= CFG.WHEAT_BUY_MAX_PRICE:
            n = wheat_target - wheat_have
            n = int(min(n, max(0, (money - CFG.CASH_RESERVE) // max(1, wprice))))
            if n > 0:
                orders.append(["BUY_PRODUCT", "WHEAT", n])
                money -= n * wprice

    # --- 5. animals FIRST -------------------------------------------------
    # Livestock has no lifespan and no production cap, so a cow bought on day 0
    # earns for 21 days while the same cow bought on day 13 earns for 8. That
    # makes early livestock the highest-compounding purchase in the game, and
    # it must outrank seed. (Buying seed first delayed our first animal to
    # day 13 and cost roughly two thirds of the season's income.)
    crop_room = {}
    for crop, target, last_day in (
            ("WHEAT", CFG.WHEAT_TILES, CFG.LAST_DAY_PLANT_WHEAT),
            ("MELON", CFG.MELON_TILES, CFG.LAST_DAY_PLANT_MELON),
            ("STRAWBERRY", CFG.STRAWBERRY_TILES, CFG.LAST_DAY_PLANT_STRAWBERRY)):
        gap = target - farm.crop_tiles.get(crop, 0) - int(farm.seeds.get(crop, 0))
        if day <= last_day and gap > 0:
            crop_room[crop] = gap

    placed = farm.n_animals
    held = sum(farm.counts.get(k, 0) for k in ANIMALS)
    unplaced = held - placed
    wprice = max(25, farm.price("WHEAT"))
    spare = money - CFG.CASH_RESERVE - held * wprice * 2
    labour_room = int(CFG.ANIMALS_PER_UNIT * farm.n_units)
    # leave the crop plan room to breathe, but never starve livestock of tiles
    tile_room = (len(farm.empty) + len(farm.free_struct)
                 - min(sum(crop_room.values()), len(farm.empty) // 2))
    # don't buy livestock faster than we can house it
    housing = len(farm.free_struct) + CFG.ANIMAL_BACKLOG - unplaced
    room = max(0, min(labour_room - held, tile_room, housing,
                      CFG.MAX_BUY_PER_DAY))

    # Size each species to the town's appetite for its product, then buy the
    # one furthest below its target. Piling into one product destroys its own
    # price (wool dies after 59 units of glut, milk after 76).
    ranked = []
    for name in CFG.SPECIES_ORDER:
        a = ANIMALS[name]
        if day > CFG.LAST_DAY_BUY.get(name, 20):
            continue
        product = a["product"]
        base = MARKET_PARAMS[product]["base"]
        ratio = farm.price(product) / float(base)
        if ratio < CFG.ANIMAL_BUY_PRICE_FRAC:
            continue
        yield_per_day = (1.0 + a["interval"]) / float(a["interval"])
        demand = drain_per_day(product, farm.shops, day, full=True)
        target = demand / yield_per_day * CFG.SUPPLY_RATIO.get(product, 1.0)
        have = farm.counts.get(name, 0)
        if have >= target:
            continue
        # rank by how far below target we are, weighted by how well the
        # product is currently clearing
        ranked.append(((target - have) * ratio, name))
    ranked.sort(reverse=True)

    for _, name in ranked:
        if room <= 0 or spare <= 0:
            break
        a = ANIMALS[name]
        # charge each purchase its own feed buffer as well as its price
        eff = a["cost"] + CFG.FEED_BUFFER_PER_ANIMAL
        n = int(min(room, spare // eff, max(1, CFG.MAX_BUY_PER_DAY // 2)))
        if n > 0:
            orders.append(["BUY_ANIMAL", name, n])
            spare -= n * eff
            room -= n

    # --- 6. seeds ---------------------------------------------------------
    # Whatever livestock left on the table goes into crops. Melon is the best
    # return per dollar ($80 of seed -> ~6 melon at ~$250 by day 10) and
    # strawberry is the best sustained lane, but both pay nothing before day 10
    # so they cannot be allowed to crowd out the early herd.
    seed_budget = max(0.0, spare) * CFG.SEED_BUDGET_FRAC
    # only buy seed we have somewhere to put -- a seed in the bag earns nothing
    tiles_free = len(farm.empty) + CFG.SEED_TILE_LOOKAHEAD
    for c in CROPS:
        tiles_free -= int(farm.seeds.get(c, 0))

    # Labour capacity gate. Planting past what the crew can water just makes
    # weeds: an unwatered plant dies after two days, and everything sown on the
    # same day dries out on the same day.
    capacity = farm.n_units * TURNS_PER_DAY * CFG.PRODUCTIVE_FRAC
    committed = (farm.n_animals * CFG.ACTIONS_PER_ANIMAL
                 + len(farm.plants) * CFG.ACTIONS_PER_PLANT)
    headroom = capacity * CFG.CAPACITY_SLACK - committed
    tiles_free = min(tiles_free, int(max(0, headroom / CFG.ACTIONS_PER_PLANT)))
    for crop in ("WHEAT", "MELON", "STRAWBERRY"):
        gap = crop_room.get(crop, 0)
        if gap <= 0 or tiles_free <= 0:
            continue
        cost = CROPS[crop]["seed"]
        n = int(min(gap, tiles_free, seed_budget // cost))
        if n > 0:
            orders.append(["BUY_SEED", crop, n])
            seed_budget -= n * cost
            spare -= n * cost
            tiles_free -= n

    return orders[:MAX_MARKET_ORDERS]


# ----------------------------------------------------------------------------
# Job generation
# ----------------------------------------------------------------------------
# A job is a dict:
#   x, y     target tile
#   op       action list to emit when standing there
#   value    $ gained by doing it (used for scheduling)
#   needs    optional item the acting unit must be carrying
#   key      dedup key so two units never take the same job

def _plant_jobs(farm, jobs):
    for (x, y, t) in farm.plants:
        crop = t.get("crop")
        cd = CROPS.get(crop)
        if not cd:
            continue
        prod_price = farm.price(cd["product"])
        age = farm.day - int(t.get("planted_day", farm.day))
        yu = int(t.get("yield_units", 0))
        unwatered = int(t.get("consecutive_unwatered", 0))
        watered = bool(t.get("watered_today", False))
        fert_until = int(t.get("fertilized_until_day", -1))

        # ---- WATER -----------------------------------------------------
        if not watered:
            value = 0.0
            if not cd["ongoing"]:
                ws = (cd["max_yield_day"] + 1) // 2
                if ws <= age <= cd["max_yield_day"] and yu < cd["max_yield"]:
                    inc = 2 if fert_until >= farm.day else 1
                    inc = min(inc, cd["max_yield"] - yu)
                    value += inc * prod_price
            else:
                if fert_until >= farm.day:
                    value += prod_price * 0.5
            if unwatered >= 1:
                # dies tonight -> value is the whole remaining plant
                remain = max(yu, cd["max_yield"] * 0.7)
                value += remain * prod_price
            else:
                # Insurance watering. A plant only has to miss two days in a
                # row to become a weed, and demand spikes because everything
                # planted together dries out together. Watering early costs an
                # otherwise-idle action and smooths that spike flat.
                value += prod_price * CFG.WATER_INSURANCE
            if value > 0:
                jobs.append({"x": x, "y": y, "op": ["WATER"], "value": value,
                             "key": ("W", x, y)})

        # ---- HARVEST ---------------------------------------------------
        if yu > 0:
            ready = False
            if cd["ongoing"]:
                ready = True
            elif age >= cd["max_yield_day"] or yu >= cd["max_yield"]:
                ready = True

            # Standing crop is free storage; the 100-item shed is not. Sixteen
            # ripe melons is 96 units landing at once, which overflows the shed
            # and forces a panic dump that took melon from $250 to $31. A
            # one-time crop holds its yield until decay, so we harvest only what
            # the shed and the market can actually take today.
            mls = int(t.get("max_lifespan_step", -1))
            urgent = (farm.day >= LAST_DAY - 1 or (0 <= mls <= farm.step))
            if ready and not urgent and not cd["ongoing"]:
                if not farm.can_absorb(cd["product"], yu):
                    ready = False

            if ready:
                jobs.append({"x": x, "y": y, "op": ["HARVEST"],
                             "value": yu * prod_price, "key": ("H", x, y)})

        # ---- FERTILIZE -------------------------------------------------
        # For ONGOING crops this is the big one: a production that is both
        # fertilized and watered yields 2 instead of 1, so keeping strawberry
        # fertilized literally doubles the tile. Fertilizer is free from the
        # animals, so this is close to pure profit.
        if fert_until < farm.day + 1:
            worth = 0.0
            if cd["ongoing"]:
                since = age - cd["first_yield_day"]
                nxt = 0 if since < 0 else (cd["interval"] - since % cd["interval"])
                if since < 0:
                    nxt = cd["first_yield_day"] - age
                if nxt <= 2:                       # a production lands in-window
                    worth = prod_price * 1.6
            else:
                ws = (cd["max_yield_day"] + 1) // 2
                if ws <= age < cd["max_yield_day"] and yu < cd["max_yield"]:
                    worth = prod_price * 1.2
            if worth > 0:
                jobs.append({"x": x, "y": y, "op": ["FERTILIZE"],
                             "value": worth, "needs": "FERTILIZER",
                             "key": ("F", x, y)})


def _animal_jobs(farm, jobs):
    for (x, y, t) in farm.animals:
        name = t.get("animal")
        a = ANIMALS.get(name)
        if not a:
            continue
        pprice = farm.price(a["product"])
        fed = bool(t.get("fed_today", False))
        cared = bool(t.get("cared_today", False))
        unfed = int(t.get("consecutive_unfed", 0))
        yu = int(t.get("yield_units", 0))

        # FEED — costs 1 wheat from the acting unit's CARRIED inventory
        if not fed:
            value = pprice * 2.0
            if unfed >= 1:
                value += a["cost"]          # escapes tonight otherwise
            jobs.append({"x": x, "y": y, "op": ["FEED"], "value": value,
                         "needs": "WHEAT", "key": ("FE", x, y)})

        # CARE — banks +1 unit onto the next production, but only counts
        # if the animal is ALSO fed today.
        if fed and not cared:
            jobs.append({"x": x, "y": y, "op": ["CARE"], "value": pprice,
                         "key": ("C", x, y)})

        # HARVEST — value scales with the stack, so big stacks win
        if yu > 0:
            v = yu * pprice
            if yu >= a["max_held"] - 2:
                v *= 1.4                     # avoid capping out and wasting
            jobs.append({"x": x, "y": y, "op": ["HARVEST"], "value": v,
                         "key": ("AH", x, y)})

        # COLLECT_FERTILIZER — free money, one per animal per day
        if t.get("fertilizer_available"):
            jobs.append({"x": x, "y": y, "op": ["COLLECT_FERTILIZER"],
                         "value": farm.price("FERTILIZER") * 0.95,
                         "key": ("CF", x, y)})


def _build_and_plant_jobs(farm, jobs, carried_animals):
    # place bought animals onto free structures
    for (x, y, t) in farm.free_struct:
        struct = t.get("kind")
        for name, a in ANIMALS.items():
            if a["structure"] != struct:
                continue
            if carried_animals.get(name, 0) > 0:
                jobs.append({"x": x, "y": y, "op": ["PLACE", name],
                             "value": a["cost"] * 1.5, "needs": name,
                             "key": ("PL", x, y)})
                break

    # build housing for animals sitting in the shed / inventories
    homeless = {"PASTURE": 0, "COOP": 0}
    for name, a in ANIMALS.items():
        n = int(farm.shed.get(name, 0)) + carried_animals.get(name, 0)
        homeless[a["structure"]] += n
    for (x, y, t) in farm.free_struct:
        k = t.get("kind")
        if k in homeless:
            homeless[k] -= 1

    empties = sorted(farm.empty, key=lambda e: farm.port_dist(e[0], e[1]))
    used = set()
    for struct, n in homeless.items():
        if n <= 0:
            continue
        op = "BUILD_PASTURE" if struct == "PASTURE" else "BUILD_COOP"
        for e in empties:
            if n <= 0:
                break
            if e in used:
                continue
            used.add(e)
            jobs.append({"x": e[0], "y": e[1], "op": [op], "value": 620.0,
                         "key": ("B", e[0], e[1])})
            n -= 1

    # Planting layout. Wheat sits nearest the shed (it is carried back and
    # feeds animals), strawberry mid-ring (long-lived, watered often), melon
    # furthest out (it only ever needs water).
    free = [e for e in empties if e not in used]           # already port-sorted
    far = sorted(free, key=lambda e: -farm.port_dist(e[0], e[1]))

    plan = (
        ("WHEAT", CFG.WHEAT_TILES, CFG.LAST_DAY_PLANT_WHEAT, free, 130.0),
        ("MELON", CFG.MELON_TILES, CFG.LAST_DAY_PLANT_MELON, far,
         farm.price("MELON") * 3.0),
        ("STRAWBERRY", CFG.STRAWBERRY_TILES, CFG.LAST_DAY_PLANT_STRAWBERRY,
         free, farm.price("STRAWBERRY") * 4.0),
    )
    for crop, target, last_day, order, value in plan:
        if farm.day > last_day:
            continue
        seeds = int(farm.seeds.get(crop, 0))
        room = target - farm.crop_tiles.get(crop, 0)
        n = min(seeds, room)
        if n <= 0:
            continue
        for e in order:
            if n <= 0:
                break
            if e in used:
                continue
            used.add(e)
            n -= 1
            jobs.append({"x": e[0], "y": e[1], "op": ["PLANT", crop],
                         "value": value, "crop": crop,
                         "key": ("P", e[0], e[1])})

    for (x, y) in farm.weeds:
        # A weed occupies a tile that could hold a $800 strawberry, so
        # clearing it is worth far more than the action it costs.
        jobs.append({"x": x, "y": y, "op": ["DIG"], "value": CFG.DIG_VALUE,
                     "key": ("D", x, y)})


def _logistics_jobs(farm, jobs, need_wheat, carried_animals):
    """Shed trips: pick up feed wheat, pick up animals, deposit produce."""
    wheat_in_shed = int(farm.shed.get("WHEAT", 0))
    for pi, port in enumerate(farm.ports):
        if need_wheat > 0 and wheat_in_shed > 0:
            n = int(min(CFG.WHEAT_PICKUP, wheat_in_shed, need_wheat))
            jobs.append({"x": port[0], "y": port[1],
                         "op": ["PICKUP", "WHEAT", n],
                         "value": 150.0 + 40.0 * n, "key": ("PW", pi)})
        for name, a in ANIMALS.items():
            if int(farm.shed.get(name, 0)) > 0:
                # only fetch an animal if there is somewhere to put it
                free = sum(1 for (_, _, t) in farm.free_struct
                           if t.get("kind") == a["structure"])
                if free - carried_animals.get(name, 0) > 0:
                    jobs.append({"x": port[0], "y": port[1],
                                 "op": ["PICKUP", name, 1],
                                 "value": a["cost"] * 1.2,
                                 "key": ("PA", pi, name)})
        if int(farm.shed.get("FERTILIZER", 0)) > 0 and \
                farm.price("FERTILIZER") < CFG.FERT_USE_BELOW:
            jobs.append({"x": port[0], "y": port[1],
                         "op": ["PICKUP", "FERTILIZER", 3],
                         "value": 70.0, "key": ("PF", pi)})


# ----------------------------------------------------------------------------
# Assignment
# ----------------------------------------------------------------------------

def _carry_count(inv):
    return sum(int(v) for v in inv.values())


def assign(farm, jobs):
    """Greedy value-minus-travel assignment of units to jobs.

    ~16 units x ~200 jobs = 3200 candidate pairs, sorted once. Well inside
    the 1 s/turn budget (measured p99 < 6 ms).
    """
    n_units = farm.n_units
    actions = [["PASS"] for _ in range(n_units)]
    assigned = [False] * n_units
    taken = set()

    pairs = []
    for i in range(n_units):
        pos = farm.unit_pos(i)
        inv = farm.unit_inv(i)
        for j, job in enumerate(jobs):
            need = job.get("needs")
            if need and int(inv.get(need, 0)) <= 0:
                continue
            d = _dist(pos, (job["x"], job["y"]))
            score = job["value"] - CFG.TRAVEL_COST * d
            if score <= 0:
                continue
            pairs.append((score, i, j))
    pairs.sort(key=lambda p: -p[0])

    # respect the atomic-PLANT rule: requesting more plants of a crop than we
    # hold seeds cancels ALL of them, so cap plant requests at seeds held.
    seed_budget = {c: int(farm.seeds.get(c, 0)) for c in CROPS}

    for score, i, j in pairs:
        if assigned[i]:
            continue
        job = jobs[j]
        key = job["key"]
        if key in taken:
            continue
        crop = job.get("crop")
        if crop is not None:
            if seed_budget.get(crop, 0) <= 0:
                continue
        pos = farm.unit_pos(i)
        if pos[0] == job["x"] and pos[1] == job["y"]:
            actions[i] = list(job["op"])
            if crop is not None:
                seed_budget[crop] -= 1
        else:
            mv = _step_toward(pos[0], pos[1], job["x"], job["y"])
            if mv is None:
                continue
            actions[i] = [mv]
        assigned[i] = True
        taken.add(key)

    # idle units: bank what they are carrying, otherwise drift to a port
    for i in range(n_units):
        if assigned[i]:
            continue
        inv = farm.unit_inv(i)
        load = _carry_count(inv)
        pos = farm.unit_pos(i)
        if load <= 0:
            continue
        port = farm.nearest_port(pos[0], pos[1])
        if tuple(pos) == tuple(port):
            # PLACE clamps to remaining shed room; DROP would destroy overflow
            item = max(inv.items(), key=lambda kv: int(kv[1]))[0]
            actions[i] = ["PLACE", item, int(inv[item])]
        elif load >= CFG.SHED_DEPOSIT_LOAD or farm.day >= LAST_DAY:
            mv = _step_toward(pos[0], pos[1], port[0], port[1])
            if mv:
                actions[i] = [mv]
    return actions


# ----------------------------------------------------------------------------
# Entry point
# ----------------------------------------------------------------------------

def _policy(obs):
    farm = Farm(obs)
    state = _state(farm.player)

    carried_animals = {}
    for inv in farm.inventories:
        for name in ANIMALS:
            c = int(inv.get(name, 0))
            if c:
                carried_animals[name] = carried_animals.get(name, 0) + c

    # how much feed wheat still needs carrying out to the field today
    unfed = sum(1 for (_, _, t) in farm.animals if not t.get("fed_today"))
    carried_wheat = sum(int(i.get("WHEAT", 0)) for i in farm.inventories)
    need_wheat = max(0, unfed - carried_wheat)

    jobs = []
    _animal_jobs(farm, jobs)
    _plant_jobs(farm, jobs)
    _build_and_plant_jobs(farm, jobs, carried_animals)
    _logistics_jobs(farm, jobs, need_wheat, carried_animals)

    unit_actions = assign(farm, jobs)
    market = plan_market(farm, state)

    state["last_step"] = farm.step
    return {
        "farmer": unit_actions[0] if unit_actions else ["PASS"],
        "hands": unit_actions[1:],
        "market": market,
    }


# Set True when testing locally so bugs surface instead of silently degrading
# the agent to PASS. MUST be False for submission.
DEBUG_RAISE = False


def agent(obs, config=None):
    """Kaggle entry point. Never raises: a crash forfeits the whole episode."""
    try:
        return _policy(obs)
    except Exception:
        if DEBUG_RAISE:
            raise
        try:
            n = len(obs["farms"][obs.get("player", 0)].get("hands") or [])
        except Exception:
            n = 0
        return {"farmer": ["PASS"], "hands": [["PASS"]] * n, "market": []}

---
## 6. Validation

Scoring is win/loss, so we measure **win-rate with a Wilson confidence interval**, and play every
seed from **both seats** to cancel seat bias. Coins are reported only as a diagnostic.

### 6.1 The thing that matters most: does it ever crash?

Kaggle runs a self-play validation episode on upload, and an exception there marks the whole
submission `Error`. Our agent wraps everything in `try/except`, which is correct for submission but
dangerous while developing -- a bug silently degrades the agent to `PASS` instead of failing loudly.
(This bit us: a one-word `NameError` cost ~$50k of in-game income before it was spotted.) Hence
`DEBUG_RAISE`, and hence this check runs the *raw* policy.

In [ ]:
import time, traceback
import main as ag

times, errs = [], []

def probe(obs, config=None):
    t = time.perf_counter()
    try:
        r = ag._policy(obs)                 # raw policy: no exception swallowing
    except Exception:
        errs.append(traceback.format_exc())
        r = {"farmer": ["PASS"], "hands": [], "market": []}
    times.append(time.perf_counter() - t)
    return r

for seed in (1, 2, 3, 101, 777):
    env = make("kaggriculture", configuration={"seed": seed}, debug=True)
    env.run([probe, "starter"])
    assert [s.status for s in env.steps[-1]] == ["DONE", "DONE"]

times.sort(); n = len(times)
print(f"turns timed {n} | mean {1000*sum(times)/n:.2f} ms | p50 {1000*times[n//2]:.2f} ms "
      f"| p99 {1000*times[int(n*0.99)]:.2f} ms | max {1000*times[-1]:.2f} ms")
print(f"policy time per episode: {sum(times)/5:.2f} s   (budget: 720 x 1s + 60s overage)")
print("raw-policy exceptions:", len(errs))
if errs:
    print(errs[0][:800])

In [ ]:
# the exact self-play episode Kaggle runs on upload
env = make("kaggriculture", configuration={"seed": 424242}, debug=True)
env.run(["main.py", "main.py"])
print("statuses:", [s.status for s in env.steps[-1]])
print("rewards :", [float(s.reward or 0) for s in env.steps[-1]])
assert [s.status for s in env.steps[-1]] == ["DONE", "DONE"], "self-play FAILED"
print("\nself-play validation PASSED")

Measured on this machine: **0 exceptions over 3,595 turns**, mean **0.41 ms/turn**, p99
**0.84 ms**, worst single turn **11.5 ms**. Total policy time is ~0.30 s per episode against a
budget of 720 s. Compute is a non-issue; there is plenty of headroom for a smarter scheduler later.

### 6.2 Win-rate

In [ ]:
import math

def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0, c - h), min(1, c + h)


def match(a, b, games=8, seed0=500):
    score = w = l = d = 0
    mine, theirs = [], []
    for i in range(games):
        seat = i % 2                       # alternate seats to cancel seat bias
        agents = [a, b] if seat == 0 else [b, a]
        e = make("kaggriculture", configuration={"seed": seed0 + i // 2})
        e.run(agents)
        r = [float(s.reward or 0) for s in e.steps[-1]]
        ma, mb = r[seat], r[1 - seat]
        mine.append(ma); theirs.append(mb)
        if ma > mb: w += 1; score += 1
        elif ma < mb: l += 1
        else: d += 1; score += 0.5
    lo, hi = wilson(score, games)
    mine.sort(); theirs.sort()
    print(f"vs {str(b).split('/')[-1]:<18} W{w} D{d} L{l}   "
          f"win-rate {100*score/games:5.1f}%  95% CI [{100*lo:.0f}%, {100*hi:.0f}%]   "
          f"median {mine[games//2]:>9,.0f} vs {theirs[games//2]:>9,.0f}")


for opp in ("random", "starter"):
    match("main.py", opp, games=8)

Measured over 16 games each (both seats, 8 distinct seeds):

| Opponent | W-D-L | Win-rate | Median coins (us vs them) |
|---|---|---:|---|
| `random` | 16-0-0 | **100%** | 98,054 vs 0 |
| `starter` | 16-0-0 | **100%** | 90,693 vs 3,507 |
| **2651.2 replay** | **0-0-16** | **0%** | **70,822 vs 172,567** |

### 6.3 The real benchmark -- and an honest result

The public scoring notebook embeds a byte-exact replay of a leaderboard agent that scored
**2651.2**. This cell extracts it and verifies its published SHA256 before using it.

**We do not beat it yet.** v01 reaches roughly **40% of its final bank**. That is the honest
state of this submission: it is a solid, crash-free agent that dominates the built-in baselines,
and it is not yet competitive with the top of the public leaderboard.

One caveat that cuts *against* us, not for us: the benchmark is a **fixed action replay**, not a
live policy. It repeats one recorded episode regardless of what we or the market do, so it plays
*below* the real agent's strength on any seed but its own. Beating it is a necessary condition,
not a sufficient one.

In [ ]:
import ast, hashlib, glob

def extract_trace_agent():
    hits = (glob.glob("**/2651*scoring*.ipynb", recursive=True)
            + glob.glob("/kaggle/input/**/2651*scoring*.ipynb", recursive=True))
    if not hits:
        return None
    nb = json.load(open(hits[0]))
    for cell in nb["cells"]:
        s = "".join(cell.get("source", []))
        if "AGENT_SOURCE" not in s:
            continue
        vals = {}
        for node in ast.parse(s).body:
            if isinstance(node, ast.Assign) and isinstance(node.value, ast.Constant):
                for t in node.targets:
                    if isinstance(t, ast.Name):
                        vals[t.id] = node.value.value
        src = vals.get("AGENT_SOURCE")
        if not src:
            continue
        exp = vals.get("EXPECTED_SHA256")
        got = hashlib.sha256(src.encode()).hexdigest()
        print("extracted:", len(src), "chars | sha256 verified:", got == exp)
        open("tran_trace.py", "w").write(src)
        return "tran_trace.py"
    return None


trace = extract_trace_agent()
if trace:
    match("main.py", trace, games=8, seed0=900)
else:
    print("2651.2 scoring notebook not found - skipping benchmark.")

### 6.4 Where the actions go

Movement is the dominant cost in this game -- the leaderboard agent spends **55%** of its unit
actions simply walking. This is our regression metric.

In [ ]:
from collections import Counter

e = make("kaggriculture", configuration={"seed": 77}, debug=True)
e.run(["main.py", "starter"])

ops, total, moves = Counter(), 0, 0
for frame in e.steps[:-1]:
    act = frame[0].action
    if not isinstance(act, dict):
        continue
    for u in [act.get("farmer")] + list(act.get("hands") or []):
        if u:
            ops[u[0]] += 1; total += 1
            if u[0] in ("NORTH", "SOUTH", "EAST", "WEST"):
                moves += 1

print("final money :", [float(s.reward or 0) for s in e.steps[-1]])
print(f"unit actions: {total}   MOVEMENT {100*moves/max(1,total):.1f}%   "
      f"(2651.2 agent: 55.0%)")
for op, n in ops.most_common():
    print(f"  {op:20} {n:5}  {100*n/max(1,total):5.1f}%")

In [ ]:
env = make("kaggriculture", configuration={"seed": 5}, debug=True)
env.run(["main.py", "starter"])
env.render(mode="ipython", width=900, height=650)

---
## 7. Build the submission

In [ ]:
import tarfile, hashlib

assert ag.DEBUG_RAISE is False, "DEBUG_RAISE must be False for submission"

with tarfile.open("submission.tar.gz", "w:gz") as t:
    t.add("main.py", arcname="main.py")

with tarfile.open("submission.tar.gz", "r:gz") as t:
    names = [m.name for m in t.getmembers()]
    packed = t.extractfile("main.py").read()

assert names == ["main.py"], names
print("bundle  : submission.tar.gz")
print("contents:", names)
print("bytes   :", len(packed))
print("sha256  :", hashlib.sha256(packed).hexdigest())

# run the BUNDLED copy, so we ship exactly what we tested
open("_verify_main.py", "wb").write(packed)
e = make("kaggriculture", configuration={"seed": 4242})
e.run(["_verify_main.py", "starter"])
print("bundled copy:", [s.status for s in e.steps[-1]],
      [float(s.reward or 0) for s in e.steps[-1]])

```bash
kaggle competitions submit kaggriculture -f submission.tar.gz -m "v01 drip"
```

---
## 8. What this agent gets wrong (the v02 backlog)

Every one of these was measured, not guessed. The numbers are median final bank against the
2651.2 replay on a fixed seed set.

**Bugs found and fixed while building this** -- worth recording because they are not obvious from
the docs and each was worth more than any parameter tweak:

| Bug | Cost | Fix |
|---|---|---|
| Feed wheat hoarded in the shed | **$14k -> $45k** | The shed holds **100 items total**, shared between feed and produce, and end-of-day overflow is **destroyed**. A 3-day feed buffer for 32 animals permanently filled it and binned every harvest. Buy wheat just-in-time. |
| Bought seed before livestock | $46k -> $57k | Animals have no lifespan: a cow bought day 0 earns for 21 days, day 13 for 8. First animal was arriving on **day 13**. |
| Harvest gated on sell price | $60k -> $71k | With the floor at base, ripe melons were never harvested and **rotted on the plant**; income did not start until day 21. |
| 36 geese bought, 2 placed | ~$10.8k wasted | Livestock purchases now capped by available housing. |
| Herd sized to labour, not demand | milk $160 -> $24 | 28 cows gluts a market that only drains 26/day. |

**Still open:**

| Gap | Evidence | Next step |
|---|---|---|
| Movement dominates | ~50-60% of actions are walking | Persistent per-unit territories; Hungarian assignment instead of greedy |
| Under-producing | Every episode ends with 60-88 **empty tiles** and cash idle in the bank | The bottleneck is mid-game cash conversion, not tiles or market depth |
| Late-game reinvestment | $44k banked at day 24 with 76 tiles free | Keep planting/buying later; the windows are still too conservative |
| Fertilizer under-collected | ~30% of available units collected | It is free money with zero town competition -- raise its job value |
| Parameters barely tuned | Hand-swept one axis at a time | Self-play CMA-ES over `CFG` (every knob is already an env-var hook) |
| No opponent modelling | Premium lanes are a shared commons | The opponent's farm is public -- read it and pre-empt |

**Submission discipline:** only your latest **2** submissions stay active. Keep one known-good
agent live at all times and never burn both slots on untested variants, especially near the
deadline.